In [2]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.1 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

In [4]:
# --- Load Kaggle data ---
df = pd.read_csv("/content/train.csv")
df.shape


(58645, 13)

In [ ]:
df.head(15)

,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
0,0,37,35000,RENT,0.0,EDUCATION,B,6000,11.49,0.17,N,14,0
1,1,22,56000,OWN,6.0,MEDICAL,C,4000,13.35,0.07,N,2,0
2,2,29,28800,OWN,8.0,PERSONAL,A,6000,8.90,0.21,N,10,0
3,3,30,70000,RENT,14.0,VENTURE,B,12000,11.11,0.17,N,5,0
4,4,22,60000,RENT,2.0,MEDICAL,A,6000,6.92,0.10,N,3,0
5,5,27,45000,RENT,2.0,VENTURE,A,9000,8.94,0.20,N,5,0
6,6,25,45000,MORTGAGE,9.0,EDUCATION,A,12000,6.54,0.27,N,3,0
7,7,21,20000,RENT,0.0,PERSONAL,C,2500,13.49,0.13,Y,3,0
8,8,37,69600,RENT,11.0,EDUCATION,D,5000,14.84,0.07,Y,11,0
9,9,35,110000,MORTGAGE,0.0,DEBTCONSOLIDATION,C,15000,12.98,0.14,Y,6,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58645 entries, 0 to 58644
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          58645 non-null  int64  
 1   person_age                  58645 non-null  int64  
 2   person_income               58645 non-null  int64  
 3   person_home_ownership       58645 non-null  object 
 4   person_emp_length           58645 non-null  float64
 5   loan_intent                 58645 non-null  object 
 6   loan_grade                  58645 non-null  object 
 7   loan_amnt                   58645 non-null  int64  
 8   loan_int_rate               58645 non-null  float64
 9   loan_percent_income         58645 non-null  float64
 10  cb_person_default_on_file   58645 non-null  object 
 11  cb_person_cred_hist_length  58645 non-null  int64  
 12  loan_status                 58645 non-null  int64  
dtypes: float64(3), int64(6), object

In [ ]:
df.loan_intent.value_counts()

,count
loan_intent,
EDUCATION,12271
MEDICAL,10934
PERSONAL,10016
VENTURE,10011
DEBTCONSOLIDATION,9133
HOMEIMPROVEMENT,6280


In [ ]:
df.person_home_ownership.value_counts()

,count
person_home_ownership,
RENT,30594
MORTGAGE,24824
OWN,3138
OTHER,89


In [5]:
# 2. Basic Cleaning: Check for nulls and drop ID
df.drop(columns=['id'], inplace=True)
# 3. Feature Engineering: Binary Encoding
# Convert 'cb_person_default_on_file' (Y/N) to (1/0)
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})
# 4. Ordinal Encoding for Loan Grade
# Grades A-D have a natural hierarchy
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
df['loan_grade'] = df['loan_grade'].map(grade_map)
# 5. One-Hot Encoding for Nominal Categories
# For Home Ownership and Loan Intent (no inherent order)
df = pd.get_dummies(df, columns=['person_home_ownership',
                                 'loan_intent'], prefix=['home', 'intent'])
# Final Output
print("--- Preprocessed Dataset Head ---")
print(df.head())
print("\n--- Data Info ---")
print(df.info())

--- Preprocessed Dataset Head ---
   person_age  person_income  person_emp_length  loan_grade  loan_amnt  \
0          37          35000                0.0         2.0       6000   
1          22          56000                6.0         3.0       4000   
2          29          28800                8.0         1.0       6000   
3          30          70000               14.0         2.0      12000   
4          22          60000                2.0         1.0       6000   

   loan_int_rate  loan_percent_income  cb_person_default_on_file  \
0          11.49                 0.17                          0   
1          13.35                 0.07                          0   
2           8.90                 0.21                          0   
3          11.11                 0.17                          0   
4           6.92                 0.10                          0   

   cb_person_cred_hist_length  loan_status  home_MORTGAGE  home_OTHER  \
0                          14          

In [ ]:
df.person_income.value_counts().sort_index()

,count
person_income,
4200,1
5000,1
9600,14
10000,1
10140,1
...,...
948000,1
1200000,2
1824000,1


In [6]:
# Log-transformation of income values (values in Thousands)
df.person_income = np.log(df.person_income)

In [7]:
# drop Nan
df.dropna(inplace=True)

In [9]:
#split data to train and test and val
from sklearn.model_selection import train_test_split
X = df.drop(columns=['loan_status'])
y = df['loan_status'].values

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:

scaler = StandardScaler()
X = scaler.fit_transform(X)

# X_test_scaled = scaler.transform(X_test)

INPUT_DIM = X.shape[1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# X = x_train_scale
# y = y_train

In [13]:
# --- Dynamic model: now works for any tabular input dim ---
def create_model(trial, input_dim):
    n_layers = trial.suggest_int("n_layers", 2, 8)
    layers = []
    in_features = input_dim

    for i in range(n_layers):
        # Fix Bug 2: use [32, 256] — clean multiple of step=32
        out_features = trial.suggest_int(f"n_units_l{i}", 4, 128, step=4)
        layers.append(nn.Linear(in_features, out_features))

        use_bn = trial.suggest_categorical(f"use_bn_l{i}", [True, False])
        if use_bn:
            layers.append(nn.BatchNorm1d(out_features))

        layers.append(nn.ReLU())
        p = trial.suggest_float(f"dropout_l{i}", 0.1, 0.5)
        layers.append(nn.Dropout(p))
        in_features = out_features

    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)


In [14]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512])
    epochs = trial.suggest_int("epochs", 10, 20)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_idx, val_idx = next(skf.split(X, y))

    X_tr = torch.tensor(X[train_idx], dtype=torch.float32)
    y_tr = torch.tensor(y[train_idx], dtype=torch.float32).unsqueeze(1)
    X_val = torch.tensor(X[val_idx], dtype=torch.float32)
    y_val = y[val_idx]

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    model = create_model(trial, INPUT_DIM).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    model.train()

    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # Pruning: apply sigmoid manually here for AUC evaluation
        model.eval()
        with torch.no_grad():
            logits = model(X_val.to(DEVICE))
            preds = torch.sigmoid(logits).cpu().numpy().ravel()

        auc = roc_auc_score(y_val, preds)
        trial.report(auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return auc

In [15]:
# --- Run the study ---
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study.optimize(objective, n_trials=50, timeout=3600)

[I 2026-04-02 23:39:25,660] A new study created in memory with name: no-name-29975f61-4a9b-44d7-a27c-3b5cdd452552
[I 2026-04-02 23:40:24,516] Trial 0 finished with value: 0.9216710697865916 and parameters: {'lr': 0.0005611516415334506, 'batch_size': 32, 'epochs': 10, 'weight_decay': 0.0005399484409787432, 'n_layers': 6, 'n_units_l0': 92, 'use_bn_l0': False, 'dropout_l0': 0.4329770563201687, 'n_units_l1': 28, 'use_bn_l1': False, 'dropout_l1': 0.2216968971838151, 'n_units_l2': 68, 'use_bn_l2': True, 'dropout_l2': 0.34474115788895177, 'n_units_l3': 20, 'use_bn_l3': False, 'dropout_l3': 0.28242799368681437, 'n_units_l4': 104, 'use_bn_l4': False, 'dropout_l4': 0.33696582754481696, 'n_units_l5': 8, 'use_bn_l5': True, 'dropout_l5': 0.12602063719411183}. Best is trial 0 with value: 0.9216710697865916.
[I 2026-04-02 23:41:34,486] Trial 1 finished with value: 0.9241641879872684 and parameters: {'lr': 0.007902619549708232, 'batch_size': 32, 'epochs': 14, 'weight_decay': 1.7541893487450798e-05, 'n

In [16]:
print("Best AUC:", study.best_value)
print("Best params:", study.best_trial.params)

Best AUC: 0.9251833277029702
Best params: {'lr': 0.0004764963542138519, 'batch_size': 128, 'epochs': 17, 'weight_decay': 1.4732618059500196e-05, 'n_layers': 3, 'n_units_l0': 116, 'use_bn_l0': True, 'dropout_l0': 0.14058861714641285, 'n_units_l1': 88, 'use_bn_l1': False, 'dropout_l1': 0.3194935157466344, 'n_units_l2': 92, 'use_bn_l2': True, 'dropout_l2': 0.3848716885390143}


In [17]:
# --- Visualize the optimization landscape ---
optuna.visualization.plot_optimization_history(study).show()
optuna.visualization.plot_param_importances(study).show()